# Local Qwen Tool Calling Test — SGLang OpenAI Server + LangGraph

This notebook is made for **Google Colab**.

It tests three things:

1. Start a local **SGLang OpenAI-compatible server** with Qwen.
2. Test direct OpenAI-style tool calling.
3. Test the same local Qwen server with **LangGraph** tools.

> Important: `api_key="EMPTY"` is only a dummy value for the OpenAI-compatible client.  
> When `base_url="http://localhost:8000/v1"`, you are calling your own local Colab server, not the paid OpenAI API.

Recommended Colab runtime: **GPU**.

## 1. Check GPU

In Colab, go to:

`Runtime` → `Change runtime type` → `T4 GPU`, `L4 GPU`, or better.

A 2B model should usually be fine on a GPU runtime, but very large context windows like `262144` may cause out-of-memory on smaller GPUs.

In [ ]:
!nvidia-smi

## 2. Install dependencies

This installs:

- `sglang` for serving the local Qwen model
- `openai` for direct OpenAI-compatible API testing
- `langchain-openai` and `langgraph` for LangGraph testing
- `pandas` for CSV tools

In [ ]:
!pip install -q --upgrade "sglang[all]>=0.4.6.post1" "openai>=1.0.0" "langchain-openai" "langgraph" "langchain-core" "pandas" "requests"

## 3. Server configuration

Default model:

```text
Qwen/Qwen3.5-2B
```

For Colab T4, start with a smaller context length like `32768`.

If you have L4/A100 or a stronger GPU, you can try:

```python
CONTEXT_LENGTH = 262144
```

But do not start with 262k context on a small GPU. It can easily crash because KV cache needs a lot of memory.

In [ ]:
MODEL_ID = "Qwen/Qwen3.5-2B"
PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# Safer for Colab T4/L4.
# You can increase this later if your GPU has enough memory.
CONTEXT_LENGTH = 32768

TP_SIZE = 1
MEM_FRACTION_STATIC = 0.75

print("MODEL_ID:", MODEL_ID)
print("BASE_URL:", BASE_URL)
print("CONTEXT_LENGTH:", CONTEXT_LENGTH)

## 4. Start SGLang server in the background

This launches Qwen as a local OpenAI-compatible server.

The important flag for tool calling is:

```bash
--tool-call-parser qwen3_coder
```

The notebook waits until `/v1/models` is reachable.

In [ ]:
import os
import sys
import time
import signal
import socket
import subprocess
from pathlib import Path

import requests

def is_port_open(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex((host, port)) == 0

# If a previous server is already running on this port, reuse it.
if is_port_open("127.0.0.1", PORT):
    print(f"Server already seems to be running on port {PORT}. Reusing it.")
    sglang_process = None
else:
    log_path = Path("/content/sglang_server.log")
    log_file = open(log_path, "w")

    cmd = [
        sys.executable, "-m", "sglang.launch_server",
        "--model-path", MODEL_ID,
        "--host", "0.0.0.0",
        "--port", str(PORT),
        "--tp-size", str(TP_SIZE),
        "--mem-fraction-static", str(MEM_FRACTION_STATIC),
        "--context-length", str(CONTEXT_LENGTH),
        "--tool-call-parser", "qwen3_coder",
    ]

    print("Starting server:")
    print(" ".join(cmd))
    print(f"Logs: {log_path}")

    sglang_process = subprocess.Popen(
        cmd,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        preexec_fn=os.setsid,
    )

# Wait for the OpenAI-compatible endpoint.
ready = False
for i in range(180):
    try:
        r = requests.get(f"{BASE_URL}/models", timeout=2)
        if r.status_code == 200:
            ready = True
            print("SGLang server is ready.")
            print(r.json())
            break
    except Exception:
        pass

    if i % 10 == 0:
        print(f"Waiting for server... {i}s")
    time.sleep(1)

if not ready:
    print("Server did not become ready. Showing last log lines:")
    log_path = Path("/content/sglang_server.log")
    if log_path.exists():
        print("\n".join(log_path.read_text(errors="ignore").splitlines()[-80:]))
    raise RuntimeError("SGLang server failed to start.")

## 5. Direct OpenAI-compatible tool calling test

This test uses the `openai` Python package, but it does **not** call OpenAI.

It calls:

```text
http://localhost:8000/v1/chat/completions
```

The API key is a dummy placeholder.

In [ ]:
from openai import OpenAI
import json

client = OpenAI(
    base_url=BASE_URL,
    api_key="EMPTY",
)

def add_numbers(a: float, b: float) -> dict:
    return {"result": a + b}

available_functions = {
    "add_numbers": add_numbers,
}

tools = [
    {
        "type": "function",
        "function": {
            "name": "add_numbers",
            "description": "Add two numbers and return the result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {
                        "type": "number",
                        "description": "First number"
                    },
                    "b": {
                        "type": "number",
                        "description": "Second number"
                    },
                },
                "required": ["a", "b"],
            },
        },
    }
]

messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant. Use tools when calculation is needed."
    },
    {
        "role": "user",
        "content": "What is 123.5 + 876.25? Use the tool."
    },
]

first_response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    tools=tools,
    tool_choice="auto",
    temperature=0,
)

assistant_message = first_response.choices[0].message
print("Assistant message:")
print(assistant_message)

if assistant_message.tool_calls:
    messages.append(assistant_message.model_dump(exclude_none=True))

    for tool_call in assistant_message.tool_calls:
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)

        print("\nTool call detected:")
        print("name:", function_name)
        print("args:", function_args)

        function_result = available_functions[function_name](**function_args)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(function_result),
        })

    final_response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        tools=tools,
        temperature=0,
    )

    print("\nFinal answer:")
    print(final_response.choices[0].message.content)
else:
    print("\nNo tool call detected. Raw answer:")
    print(assistant_message.content)

## 6. LangGraph simple tool-calling test

Now we connect LangGraph to the same local Qwen server.

Flow:

```text
LangGraph → ChatOpenAI wrapper → local SGLang server → Qwen → tool call → Python tool → final answer
```

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

@tool
def multiply_numbers(a: float, b: float) -> dict:
    """Multiply two numbers and return the result."""
    return {"result": a * b}

llm = ChatOpenAI(
    model=MODEL_ID,
    base_url=BASE_URL,
    api_key="EMPTY",
    temperature=0,
)

simple_agent = create_react_agent(
    model=llm,
    tools=[multiply_numbers],
)

result = simple_agent.invoke({
    "messages": [
        HumanMessage(content="What is 37 multiplied by 19? Use the tool.")
    ]
})

for message in result["messages"]:
    message.pretty_print()

# 7. CSV tool-calling agent

This section creates practical tools for a CSV question-answering app.

You can upload any CSV file. The agent gets tools like:

- inspect dataset schema
- resolve approximate column names
- sort rows
- group by count
- summarize numeric columns
- filter rows by value

This is closer to the application you want to build.

In [ ]:
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded.")

csv_filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[csv_filename]))

print("Loaded:", csv_filename)
print("Shape:", df.shape)
df.head()

## 8. Define CSV tools

These tools are intentionally small and strict.

That is better than one giant `analyze_csv()` tool because small tools are easier for the model to call correctly.

In [ ]:
from typing import Optional
from difflib import get_close_matches
import pandas as pd
import json

def _safe_preview(dataframe: pd.DataFrame, max_rows: int = 10) -> str:
    """Return a compact markdown preview."""
    if dataframe.empty:
        return "No rows found."
    return dataframe.head(max_rows).to_markdown(index=False)

def _resolve_column(column_hint: str) -> str:
    """Resolve approximate column name to actual dataframe column."""
    columns = list(df.columns)

    if column_hint in columns:
        return column_hint

    lower_map = {c.lower().strip(): c for c in columns}
    key = column_hint.lower().strip()
    if key in lower_map:
        return lower_map[key]

    matches = get_close_matches(column_hint, columns, n=1, cutoff=0.4)
    if matches:
        return matches[0]

    lower_matches = get_close_matches(key, list(lower_map.keys()), n=1, cutoff=0.4)
    if lower_matches:
        return lower_map[lower_matches[0]]

    raise ValueError(f"Could not resolve column '{column_hint}'. Available columns: {columns}")

@tool
def get_dataset_schema(sample_rows: int = 3, unique_sample_limit: int = 5) -> str:
    """
    Inspect the loaded CSV and return columns, dtypes, numeric/categorical groups,
    sample values, and sample rows.
    """
    sample_rows = max(1, min(int(sample_rows), 10))
    unique_sample_limit = max(1, min(int(unique_sample_limit), 20))

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    categorical_cols = [c for c in df.columns if c not in numeric_cols]

    column_info = []
    for col in df.columns:
        non_null = int(df[col].notna().sum())
        null_count = int(df[col].isna().sum())
        unique_count = int(df[col].nunique(dropna=True))
        samples = df[col].dropna().astype(str).unique().tolist()[:unique_sample_limit]

        column_info.append({
            "column": col,
            "dtype": str(df[col].dtype),
            "non_null": non_null,
            "null_count": null_count,
            "unique_count": unique_count,
            "sample_values": samples,
        })

    result = {
        "shape": {"rows": int(df.shape[0]), "columns": int(df.shape[1])},
        "numeric_columns": numeric_cols,
        "categorical_or_other_columns": categorical_cols,
        "columns": column_info,
        "sample_rows": df.head(sample_rows).to_dict(orient="records"),
    }
    return json.dumps(result, indent=2, default=str)

@tool
def resolve_column_name(column_hint: str) -> str:
    """
    Resolve an approximate column hint to the closest actual CSV column name.
    Use this before tools that require exact column names.
    """
    return _resolve_column(column_hint)

@tool
def sort_rows(sort_by: str, n: int = 5, ascending: bool = False, columns_to_show: Optional[str] = None) -> str:
    """
    Sort rows by a selected column and return the top or bottom N rows.
    columns_to_show can be a comma-separated string of column names/hints, or empty for all columns.
    """
    actual_sort_col = _resolve_column(sort_by)
    n = max(1, min(int(n), 50))

    sorted_df = df.sort_values(by=actual_sort_col, ascending=bool(ascending))

    if columns_to_show:
        selected_cols = []
        for col_hint in columns_to_show.split(","):
            selected_cols.append(_resolve_column(col_hint.strip()))
        sorted_df = sorted_df[selected_cols]

    return _safe_preview(sorted_df, max_rows=n)

@tool
def groupby_count(group_by: str, top_n: int = 10) -> str:
    """
    Count rows grouped by a selected categorical column.
    Returns the most frequent groups first.
    """
    actual_col = _resolve_column(group_by)
    top_n = max(1, min(int(top_n), 50))

    result_df = (
        df.groupby(actual_col, dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(top_n)
    )
    return _safe_preview(result_df, max_rows=top_n)

@tool
def numeric_summary(column: Optional[str] = None) -> str:
    """
    Return summary statistics for one numeric column or all numeric columns.
    """
    if column:
        actual_col = _resolve_column(column)
        if not pd.api.types.is_numeric_dtype(df[actual_col]):
            raise ValueError(f"Column '{actual_col}' is not numeric.")
        summary_df = df[[actual_col]].describe().T
    else:
        summary_df = df.select_dtypes(include="number").describe().T

    if summary_df.empty:
        return "No numeric columns found."

    return summary_df.to_markdown()

@tool
def filter_rows(column: str, operator: str, value: str, n: int = 10) -> str:
    """
    Filter rows by a condition.

    Supported operators:
    - eq
    - neq
    - gt
    - gte
    - lt
    - lte
    - contains

    Example:
    column='marks', operator='gt', value='80'
    """
    actual_col = _resolve_column(column)
    operator = operator.lower().strip()
    n = max(1, min(int(n), 50))

    series = df[actual_col]

    if pd.api.types.is_numeric_dtype(series):
        try:
            typed_value = float(value)
        except Exception:
            raise ValueError(f"Column '{actual_col}' is numeric, but value '{value}' is not numeric.")
    else:
        typed_value = value

    if operator == "eq":
        mask = series.astype(str).str.lower() == str(typed_value).lower()
    elif operator == "neq":
        mask = series.astype(str).str.lower() != str(typed_value).lower()
    elif operator == "contains":
        mask = series.astype(str).str.contains(str(value), case=False, na=False)
    elif operator == "gt":
        mask = series > typed_value
    elif operator == "gte":
        mask = series >= typed_value
    elif operator == "lt":
        mask = series < typed_value
    elif operator == "lte":
        mask = series <= typed_value
    else:
        raise ValueError("Unsupported operator. Use one of: eq, neq, gt, gte, lt, lte, contains.")

    return _safe_preview(df[mask], max_rows=n)

csv_tools = [
    get_dataset_schema,
    resolve_column_name,
    sort_rows,
    groupby_count,
    numeric_summary,
    filter_rows,
]

print("CSV tools ready:")
for t in csv_tools:
    print("-", t.name)

## 9. Run CSV agent with LangGraph

Try questions like:

```text
What columns are in this dataset?
```

```text
Show me the top 5 rows by marks.
```

```text
Which category appears most often?
```

```text
Give me summary statistics for the score column.
```

```text
Filter rows where city contains Jaipur.
```

In [ ]:
csv_agent = create_react_agent(
    model=llm,
    tools=csv_tools,
)

question = "Inspect this dataset and tell me what columns are available. Then suggest 5 useful questions I can ask."

result = csv_agent.invoke({
    "messages": [
        HumanMessage(content=question)
    ]
})

for message in result["messages"]:
    message.pretty_print()

In [ ]:
# Ask your own question here

question = "Show me the top 5 rows by the most relevant score or marks column. If needed, inspect the schema first."

result = csv_agent.invoke({
    "messages": [
        HumanMessage(content=question)
    ]
})

for message in result["messages"]:
    message.pretty_print()

# 10. Direct OpenAI-style tool calling with CSV tools

This is optional.

LangGraph is usually easier for tool loops, but this cell shows how to manually handle one OpenAI-style tool call.

In [ ]:
# Convert LangChain tools to OpenAI tool schema
# This works for simple LangChain StructuredTool objects.

openai_csv_tools = []
for lc_tool in csv_tools:
    schema = lc_tool.args_schema.model_json_schema()
    openai_csv_tools.append({
        "type": "function",
        "function": {
            "name": lc_tool.name,
            "description": lc_tool.description or "",
            "parameters": schema,
        }
    })

tool_map = {t.name: t for t in csv_tools}

messages = [
    {
        "role": "system",
        "content": "You are a CSV analysis assistant. Use tools to inspect and analyze the uploaded dataframe."
    },
    {
        "role": "user",
        "content": "What are the columns and dtypes in this CSV? Use the schema tool."
    }
]

resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    tools=openai_csv_tools,
    tool_choice="auto",
    temperature=0,
)

msg = resp.choices[0].message
print("Assistant message:")
print(msg)

if msg.tool_calls:
    messages.append(msg.model_dump(exclude_none=True))

    for tool_call in msg.tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments or "{}")

        print("\nTool call:")
        print(name, args)

        tool_result = tool_map[name].invoke(args)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(tool_result),
        })

    final_resp = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        tools=openai_csv_tools,
        temperature=0,
    )

    print("\nFinal answer:")
    print(final_resp.choices[0].message.content)
else:
    print("No tool call returned.")
    print(msg.content)

# 11. Troubleshooting

## Server not starting

Open the server log:

```python
print(open("/content/sglang_server.log").read()[-5000:])
```

Common fixes:

1. Reduce context length:

```python
CONTEXT_LENGTH = 8192
```

2. Reduce static memory fraction:

```python
MEM_FRACTION_STATIC = 0.6
```

3. Restart Colab runtime and run again.

## LangGraph error: model has no `bind_tools`

Use:

```python
from langchain_openai import ChatOpenAI
```

Do not use the older `OpenAI` completion wrapper for LangGraph tool calling.

## Model gives answer without tool call

Try stronger instruction:

```text
Use the available tool. Do not calculate manually.
```

Also keep tools small and clearly described.

## Stop server

Run the cleanup cell below.

In [ ]:
# Optional cleanup cell: stop the SGLang server started by this notebook.

import os
import signal

try:
    if sglang_process is not None:
        os.killpg(os.getpgid(sglang_process.pid), signal.SIGTERM)
        print("SGLang server stopped.")
    else:
        print("No server process object found. It may have been reused or already stopped.")
except Exception as e:
    print("Could not stop server:", e)